In [ ]:
# In[1]:

import pandas as pd

# Load files
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Helper to build compact summary for datasets with cmdb_id
def compact_summary_with_names(df, name_field=None):
    total_rows = len(df)
    min_ts = int(df['timestamp'].min()) if total_rows > 0 else None
    max_ts = int(df['timestamp'].max()) if total_rows > 0 else None
    # parse timestamps to UTC datetimes as required by rule 10 (kept as extra column)
    df['ts_utc'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    unique_cmdb_count = int(df['cmdb_id'].nunique()) if 'cmdb_id' in df.columns else 0
    top_cmdb_series = df['cmdb_id'].value_counts().head(20)
    top_cmdb_list = top_cmdb_series.index.tolist()
    # Build one-row summary DataFrame (keep fields compact)
    summary = pd.DataFrame([{
        'rows': total_rows,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts,
        'unique_cmdb_count': unique_cmdb_count,
        'top_cmdb_ids_topN': top_cmdb_list
    }])
    # If a name_field is provided, also return top name counts (top 20)
    top_names = None
    unique_name_count = None
    if name_field is not None and name_field in df.columns:
        name_counts = df[name_field].value_counts()
        unique_name_count = int(name_counts.size)
        top_names = name_counts.head(20).rename_axis(name_field).reset_index(name='count')
    return summary, top_names, unique_name_count

# Metric summary (kpi_name)
metric_summary, top_kpi_df, metric_unique_kpi_count = compact_summary_with_names(df_metric, 'kpi_name')

# Trace summary (trace_name)
trace_summary, top_trace_df, trace_unique_count = compact_summary_with_names(df_trace, 'trace_name')

# Log summary (log_name)
log_summary, top_log_df, log_unique_count = compact_summary_with_names(df_log, 'log_name')

# Error logs summary (messages)
total_rows_e = len(df_error)
min_ts_e = int(df_error['timestamp'].min()) if total_rows_e > 0 else None
max_ts_e = int(df_error['timestamp'].max()) if total_rows_e > 0 else None
df_error['ts_utc'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)
unique_cmdb_count_e = int(df_error['cmdb_id'].nunique()) if 'cmdb_id' in df_error.columns else 0
top_cmdb_error = df_error['cmdb_id'].value_counts().head(20)
top_cmdb_error_list = top_cmdb_error.index.tolist()
unique_message_count = int(df_error['message'].nunique()) if 'message' in df_error.columns else 0
top_message_samples = df_error['message'].value_counts().head(20).rename_axis('message').reset_index(name='count')

error_summary = pd.DataFrame([{
    'rows': total_rows_e,
    'min_timestamp': min_ts_e,
    'max_timestamp': max_ts_e,
    'unique_cmdb_count': unique_cmdb_count_e,
    'top_cmdb_ids_topN': top_cmdb_error_list,
    'unique_message_count': unique_message_count
}])

# Final compact outputs (only the aggregates requested). Display limited top-N tables where applicable.
metric_summary, top_kpi_df, trace_summary, top_trace_df, log_summary, top_log_df, error_summary, top_message_samples

```
Out[1]:
```
```python
# Build a plain-English compact summary string using previously loaded summary variables
metric_row = metric_summary.iloc[0]
trace_row = trace_summary.iloc[0]
log_row = log_summary.iloc[0]
error_row = error_summary.iloc[0]

# Prepare top lists (limit for compactness)
def top_list_from_df(df, name_col, count_col, limit=8):
    if df is None or df.empty:
        return "None"
    rows = df.head(limit)
    return "; ".join(f"{r[name_col]} ({int(r[count_col])})" for _, r in rows.iterrows())

top_kpis = top_list_from_df(top_kpi_df, 'kpi_name', 'count', limit=8)
top_traces = top_list_from_df(top_trace_df, 'trace_name', 'count', limit=8)
top_logs = top_list_from_df(top_log_df, 'log_name', 'count', limit=8)

summary_text = (
    "Compact telemetry file summary:\n\n"
    "metric.csv:\n"
    f"- total rows: {int(metric_row['rows'])}\n"
    f"- min_timestamp (unix sec): {int(metric_row['min_timestamp'])}\n"
    f"- max_timestamp (unix sec): {int(metric_row['max_timestamp'])}\n"
    f"- unique cmdb_id count: {int(metric_row['unique_cmdb_count'])}\n"
    f"- top cmdb_ids (sample up to 20): {metric_row['top_cmdb_ids_topN'][:20]}\n"
    f"- unique kpi_name count: {int(metric_unique_kpi_count)}\n"
    f"- top kpi_name (top 8 shown): {top_kpis}\n\n"
    "trace.csv:\n"
    f"- total rows: {int(trace_row['rows'])}\n"
    f"- min_timestamp (unix sec): {int(trace_row['min_timestamp'])}\n"
    f"- max_timestamp (unix sec): {int(trace_row['max_timestamp'])}\n"
    f"- unique cmdb_id count: {int(trace_row['unique_cmdb_count'])}\n"
    f"- top cmdb_ids (sample up to 20): {trace_row['top_cmdb_ids_topN'][:20]}\n"
    f"- unique trace_name count: {int(trace_unique_count)}\n"
    f"- top trace_name (top 8 shown): {top_traces}\n\n"
    "log.csv:\n"
    f"- total rows: {int(log_row['rows'])}\n"
    f"- min_timestamp (unix sec): {int(log_row['min_timestamp'])}\n"
    f"- max_timestamp (unix sec): {int(log_row['max_timestamp'])}\n"
    f"- unique cmdb_id count: {int(log_row['unique_cmdb_count'])}\n"
    f"- top cmdb_ids (sample up to 20): {log_row['top_cmdb_ids_topN'][:20]}\n"
    f"- unique log_name count: {int(log_unique_count)}\n"
    f"- top log_name (top 8 shown): {top_logs}\n\n"
    "error_logs.csv:\n"
    f"- total rows: {int(error_row['rows'])}\n"
    f"- min_timestamp (unix sec): {error_row['min_timestamp']}\n"
    f"- max_timestamp (unix sec): {error_row['max_timestamp']}\n"
    f"- unique cmdb_id count: {int(error_row['unique_cmdb_count'])}\n"
    f"- unique message count: {int(error_row['unique_message_count'])}\n"
    "- top message samples: None (no error log messages present)\n\n"
    "Notes:\n"
    "- Timestamps are raw Unix seconds (UTC).\n"
    "- Top-name lists are truncated for compactness; full lists are stored in the loaded variables."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(   rows  min_timestamp  max_timestamp  unique_cmdb_count                                  top_cmdb_ids_topN
0  9225     1705917360     1705918800                 68  [ts-admin-travel-service, ts-food-service, ts-...,      kpi_name  count
0         cpu   1700
1         mem   1700
2      socket   1700
3      diskio   1675
4  latency-90    700
5  latency-50    700
6    workload    700
7       error    350,     rows  min_timestamp  max_timestamp  unique_cmdb_count                                  top_cmdb_ids_topN
0  18648     1705917360     1705918800                 28  [root, ts-preserve-service, ts-travel-service,...,                                            trace_name  count
0                           trace.from_root.row_count    347
1                          trace.from_root.error_rate    347
2                        trace.from_root.duration_p95    347
3                       trace.from_root.duration_mean    347
4            trace.from_ts-preserve-service.row_count    259
5           trace.from_ts-preserve-service.error_rate    259
6         trace.from_ts-preserve-service.duration_p95    259
7        trace.from_ts-preserve-service.duration_mean    259
8      trace.from_ts-preserve-other-service.row_count    233
9     trace.from_ts-preserve-other-service.error_rate    233
10  trace.from_ts-preserve-other-service.duration_p95    233
11  trace.from_ts-preserve-other-service.duration_...    233
12            trace.to_ts-order-service.duration_mean    174
13             trace.to_ts-order-service.duration_p95    174
14               trace.to_ts-order-service.error_rate    174
15                trace.to_ts-order-service.row_count    174
16              trace.to_ts-station-service.row_count    171
17             trace.to_ts-station-service.error_rate    171
18           trace.to_ts-station-service.duration_p95    171
19          trace.to_ts-station-service.duration_mean    171,    rows  min_timestamp  max_timestamp  unique_cmdb_count                                  top_cmdb_ids_topN
0  2196     1705917360     1705918800                 46  [ts-admin-basic-info-service, ts-admin-travel-...,           log_name  count
0  log.error_count   1098
1    log.row_count   1098,    rows min_timestamp max_timestamp  unique_cmdb_count top_cmdb_ids_topN  unique_message_count
0     0          None          None                  0                []                     0, Empty DataFrame
Columns: [message, count]
Index: [])```
```

In [ ]:
# In[2]:

# Recompute aggregates correctly and return compact summaries
import pandas as pd
import numpy as np

# Ensure df_metric is loaded
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

components = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Filter to the five components
df_five = df_metric[df_metric['cmdb_id'].isin(components)].copy()

# Compute aggregates per (cmdb_id, kpi_name)
agg = (
    df_five
    .groupby(['cmdb_id', 'kpi_name'], sort=True, as_index=False)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        p95_value=('value', lambda s: float(np.nan) if s.empty else float(s.quantile(0.95)))
    )
)

# Ensure integer-like timestamps (nullable Int64)
agg['min_timestamp'] = agg['min_timestamp'].astype('Int64')
agg['max_timestamp'] = agg['max_timestamp'].astype('Int64')

# Sort and limit to top 50 rows
group_summary = agg.sort_values(['cmdb_id', 'kpi_name']).reset_index(drop=True).head(50)[
    ['cmdb_id','kpi_name','count','min_timestamp','max_timestamp','p95_value']
]

# Build short list of available kpi_name values per each of the five components (one line per component)
rows = []
for c in components:
    kpis = sorted(df_metric.loc[df_metric['cmdb_id']==c, 'kpi_name'].dropna().unique())
    kpi_sample = ", ".join(kpis[:20])  # sample up to 20 names
    rows.append({'cmdb_id': c, 'kpi_count': len(kpis), 'kpi_names_sample': kpi_sample})

kpi_per_component = pd.DataFrame(rows)

# Return the two compact DataFrames (IPython will display them)
group_summary, kpi_per_component

```
Out[2]:
```
```python
# Build a plain-English summary string using existing aggregated results
try:
    group_summary
    kpi_per_component
except NameError:
    # reload if needed
    import pandas as pd
    group_summary = pd.read_csv("metric.csv").query("cmdb_id in @components").groupby(['cmdb_id','kpi_name'], as_index=False).agg(count=('value','count'), min_timestamp=('timestamp','min'), max_timestamp=('timestamp','max'), p95_value=('value', lambda s: s.quantile(0.95)))
    kpi_per_component = pd.DataFrame([])

total_groups = int(group_summary.shape[0])
min_ts = int(group_summary['min_timestamp'].min()) if total_groups>0 else None
max_ts = int(group_summary['max_timestamp'].max()) if total_groups>0 else None
count_values = sorted(group_summary['count'].unique().tolist())
example_rows = group_summary.head(5).to_dict(orient='records')

# Build per-component kpi lines
per_comp_lines = []
for _, r in kpi_per_component.iterrows():
    per_comp_lines.append(f"- {r['cmdb_id']}: {r['kpi_count']} kpi(s) — {r['kpi_names_sample']}")

summary_text = (
    f"Summary of metric.csv for the five components:\n\n"
    f"- Aggregated (cmdb_id, kpi_name) groups returned: {total_groups} (limited to <=50 rows).\n"
    f"- Sample counts per group (unique counts observed): {count_values} (most groups have 25 samples).\n"
    f"- Timestamp range across groups (unix seconds, UTC): min={min_ts}, max={max_ts}.\n\n"
    f"- Example aggregated rows (up to 5):\n"
)

for ex in example_rows:
    summary_text += (
        f"  * {ex['cmdb_id']} / {ex['kpi_name']}: count={int(ex['count'])}, "
        f"min_ts={int(ex['min_timestamp'])}, max_ts={int(ex['max_timestamp'])}, "
        f"p95_value={float(ex['p95_value'])}\n"
    )

summary_text += "\n- Available kpi_name values per component:\n" + "\n".join(per_comp_lines) + "\n\n"
summary_text += "Notes: All timestamps are raw Unix seconds (UTC). The detailed aggregated table and full lists are available in `group_summary` and `kpi_per_component` variables."

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  count  min_timestamp  max_timestamp     p95_value
0     ts-auth-service         cpu     25     1705917360     1705918800  6.957539e+01
1     ts-auth-service      diskio     25     1705917360     1705918800  6.885065e+04
2     ts-auth-service  latency-50     25     1705917360     1705918800  3.851742e-01
3     ts-auth-service  latency-90     25     1705917360     1705918800  7.228632e-01
4     ts-auth-service         mem     25     1705917360     1705918800  2.619205e+08
..                ...         ...    ...            ...            ...           ...
32  ts-travel-service  latency-50     25     1705917360     1705918800  1.860750e-02
33  ts-travel-service  latency-90     25     1705917360     1705918800  2.566300e-01
34  ts-travel-service         mem     25     1705917360     1705918800  2.882683e+08
35  ts-travel-service      socket     25     1705917360     1705918800  1.938000e+01
36  ts-travel-service    workload     25     1705917360     1705918800  1.075506e+01

[37 rows x 6 columns],              cmdb_id  kpi_count                                   kpi_names_sample
0    ts-auth-service          7  cpu, diskio, latency-50, latency-90, mem, sock...
1   ts-order-service          8  cpu, diskio, error, latency-50, latency-90, me...
2   ts-route-service          7  cpu, diskio, latency-50, latency-90, mem, sock...
3   ts-train-service          7  cpu, diskio, latency-50, latency-90, mem, sock...
4  ts-travel-service          8  cpu, diskio, error, latency-50, latency-90, me...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Ensure metric and group_summary are loaded in the kernel
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

try:
    group_summary
except NameError:
    # compute global p95 per (cmdb_id,kpi_name) for the five components
    comps = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
    df_five_all = df_metric[df_metric['cmdb_id'].isin(comps)].copy()
    group_summary = (
        df_five_all
        .groupby(['cmdb_id','kpi_name'], as_index=False, sort=True)
        .agg(p95_value=('value', lambda s: float(s.quantile(0.95))))
    )

# Define components and incident window (UTC)
components = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
start_ts = pd.Timestamp("2024-01-22 09:53:25", tz="UTC")
end_ts   = pd.Timestamp("2024-01-22 10:23:25", tz="UTC")

# Ensure timestamp datetime column in UTC for filtering
if 'ts_utc' not in df_metric.columns:
    df_metric['ts_utc'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Filter rows to the five components and the incident window (inclusive)
df_window = df_metric[
    df_metric['cmdb_id'].isin(components) &
    (df_metric['ts_utc'] >= start_ts) &
    (df_metric['ts_utc'] <= end_ts)
].copy()

# Merge global p95_value from group_summary
p95_map = group_summary[['cmdb_id','kpi_name','p95_value']].drop_duplicates()
df_window = df_window.merge(p95_map, on=['cmdb_id','kpi_name'], how='left')

# Mark anomalies where value >= global p95_value (only when p95_value is present)
df_window['is_anomaly'] = False
mask_valid = df_window['p95_value'].notna()
df_window.loc[mask_valid, 'is_anomaly'] = df_window.loc[mask_valid, 'value'] >= df_window.loc[mask_valid, 'p95_value']

# Aggregate per (cmdb_id, kpi_name) within window by manual loop for robustness
rows = []
grouped = df_window.groupby(['cmdb_id','kpi_name'], sort=False)
for (cmdb, kpi), g in grouped:
    cnt = int(g.shape[0])
    an_cnt = int(g['is_anomaly'].sum())
    earliest_anom_ts = int(g.loc[g['is_anomaly'], 'timestamp'].min()) if an_cnt > 0 else pd.NA
    maxv = float(g['value'].max()) if cnt>0 else np.nan
    p95v = float(g['p95_value'].iloc[0]) if pd.notna(g['p95_value'].iloc[0]) else np.nan
    ratio = (float(maxv) / float(p95v)) if (pd.notna(p95v) and p95v != 0) else pd.NA
    rows.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'count_in_window': cnt,
        'anomaly_count': an_cnt,
        'earliest_anomaly_timestamp': earliest_anom_ts,
        'max_value_in_window': maxv,
        'p95_value': p95v,
        'ratio': ratio
    })

agg_window = pd.DataFrame(rows)

# Normalize types and sort by ratio desc then anomaly_count desc
if not agg_window.empty:
    agg_window['earliest_anomaly_timestamp'] = agg_window['earliest_anomaly_timestamp'].astype('Int64')
    # Place NaN ratios at the end by converting to float and using sort_values with na_position='last'
    agg_window['ratio_float'] = agg_window['ratio'].apply(lambda x: float(x) if pd.notna(x) else np.nan)
    agg_window_sorted = agg_window.sort_values(by=['ratio_float','anomaly_count'], ascending=[False, False], na_position='last').drop(columns=['ratio_float']).reset_index(drop=True)
else:
    agg_window_sorted = agg_window.copy()

# Top 20 rows by ratio then anomaly_count
top20_by_ratio = agg_window_sorted.head(20)[
    ['cmdb_id','kpi_name','count_in_window','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','p95_value','ratio']
]

# Separate short list of groups with anomaly_count > 0 (limit 20)
anomalies_only = agg_window_sorted[agg_window_sorted['anomaly_count']>0][
    ['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp']
].head(20).reset_index(drop=True)

# Return compact outputs
top20_by_ratio, anomalies_only

```
Out[3]:
```
# Build a plain-English compact summary from the previously computed results
import pandas as pd

try:
    top20 = top20_by_ratio.copy()
    anomalies = anomalies_only.copy()
except NameError:
    # If variables are not present, abort with an informative message
    summary_text = "Required results (top20_by_ratio, anomalies_only) are not available in the kernel."
    summary_text
else:
    # Helper to convert unix-second timestamp (Int64 / NA-able) to UTC ISO string or "None"
    def ts_to_iso(ts):
        if pd.isna(ts):
            return "None"
        return pd.to_datetime(int(ts), unit='s', utc=True).isoformat()

    # Overall counts
    total_groups_with_anomalies = int(anomalies.shape[0])
    window_start = "2024-01-22 09:53:25 UTC"
    window_end   = "2024-01-22 10:23:25 UTC"

    # Top 5 by ratio (if available)
    top_n = min(5, top20.shape[0])
    top_lines = []
    for i in range(top_n):
        r = top20.iloc[i]
        line = (
            f"{i+1}. {r['cmdb_id']} / {r['kpi_name']}: ratio={r['ratio']:.3f}, "
            f"anomalies={int(r['anomaly_count'])}, count_in_window={int(r['count_in_window'])}, "
            f"earliest_anomaly_ts={ts_to_iso(r['earliest_anomaly_timestamp'])}, "
            f"max_value={r['max_value_in_window']:.6g}, p95={r['p95_value']:.6g}"
        )
        top_lines.append(line)

    # Build short list of all groups with anomaly_count>0 (limit 20) as compact lines
    anomaly_lines = []
    for _, r in anomalies.iterrows():
        anomaly_lines.append(
            f"- {r['cmdb_id']} / {r['kpi_name']}: anomalies={int(r['anomaly_count'])}, earliest={ts_to_iso(r['earliest_anomaly_timestamp'])}"
        )

    summary_text = (
        f"Incident window: {window_start} — {window_end} (inclusive, UTC)\n\n"
        f"Summary (five services inspected):\n"
        f"- Groups with anomalies (value >= global P95): {total_groups_with_anomalies} (listed below; limited to 20).\n\n"
        f"Top {top_n} groups by ratio (max_value_in_window / global P95), sorted descending:\n"
        + "\n".join(top_lines)
        + "\n\nAll groups with anomaly_count>0 (up to 20):\n"
        + ("\n".join(anomaly_lines) if anomaly_lines else "None")
        + "\n\nNotes:\n"
        "- ratio > 1 indicates the observed max in the window exceeded the global P95 for that (cmdb_id,kpi_name).\n"
        "- Timestamps shown in UTC ISO format. Counts are number of metric rows within the incident window.\n"
    )

    summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  count_in_window  anomaly_count  earliest_anomaly_timestamp  max_value_in_window     p95_value     ratio
0    ts-order-service  latency-90               25              2                  1705917360             0.288235      0.033307  8.653825
1     ts-auth-service      diskio               25              2                  1705918080        148601.864662  68850.649906  2.158322
2   ts-travel-service         cpu               25              2                  1705917600             4.479598      3.599688  1.244441
3   ts-travel-service  latency-90               25              2                  1705917420             0.286226      0.256630  1.115326
4    ts-order-service    workload               25              2                  1705917600             7.931033      7.346383  1.079583
5    ts-order-service      diskio               25              2                  1705917600         65988.555548  61664.084557  1.070129
6     ts-auth-service  latency-90               25              2                  1705918320             0.761575      0.722863  1.053553
7    ts-route-service      diskio               25              2                  1705918020         64882.940291  61627.998293  1.052816
8    ts-train-service         cpu               25              2                  1705917600             3.598671      3.424213  1.050948
9    ts-order-service         cpu               25              2                  1705917420             3.090166      2.941714  1.050464
10   ts-route-service         cpu               25              2                  1705917600             5.003713      4.803464  1.041689
11  ts-travel-service      socket               25              2                  1705917360            20.000000     19.380000  1.031992
12  ts-travel-service    workload               25              2                  1705917540            11.083383     10.755063  1.030527
13   ts-order-service      socket               25              2                  1705917360            13.628571     13.233333  1.029867
14  ts-travel-service      diskio               25              2                  1705917660         60066.286287  58463.604173  1.027413
15   ts-train-service      diskio               25              2                  1705918320         60737.679058  59278.908240  1.024609
16   ts-train-service    workload               25              2                  1705917600            23.882167     23.451933  1.018345
17   ts-order-service  latency-50               25              2                  1705917420             0.010519      0.010334  1.017845
18   ts-route-service  latency-50               25              2                  1705917600             0.003829      0.003774  1.014695
19   ts-train-service      socket               25              2                  1705918440            19.000000     18.726667  1.014596,               cmdb_id    kpi_name  anomaly_count  earliest_anomaly_timestamp
0    ts-order-service  latency-90              2                  1705917360
1     ts-auth-service      diskio              2                  1705918080
2   ts-travel-service         cpu              2                  1705917600
3   ts-travel-service  latency-90              2                  1705917420
4    ts-order-service    workload              2                  1705917600
5    ts-order-service      diskio              2                  1705917600
6     ts-auth-service  latency-90              2                  1705918320
7    ts-route-service      diskio              2                  1705918020
8    ts-train-service         cpu              2                  1705917600
9    ts-order-service         cpu              2                  1705917420
10   ts-route-service         cpu              2                  1705917600
11  ts-travel-service      socket              2                  1705917360
12  ts-travel-service    workload              2                  1705917540
13   ts-order-service      socket              2                  1705917360
14  ts-travel-service      diskio              2                  1705917660
15   ts-train-service      diskio              2                  1705918320
16   ts-train-service    workload              2                  1705917600
17   ts-order-service  latency-50              2                  1705917420
18   ts-route-service  latency-50              2                  1705917600
19   ts-train-service      socket              2                  1705918440)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import re

# Ensure df_trace is loaded (reuse if present)
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Compute global P95 per trace_name across the full file (rule 9)
p95_trace = (
    df_trace
    .groupby('trace_name', sort=False, as_index=False)
    .agg(p95_value=('value', lambda s: float(s.quantile(0.95))))
)

# Prepare timestamps in UTC for filtering (rule 10)
if 'ts_utc' not in df_trace.columns:
    df_trace['ts_utc'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start_ts = pd.Timestamp("2024-01-22 09:53:25", tz="UTC")
end_ts   = pd.Timestamp("2024-01-22 10:23:25", tz="UTC")

# Filter rows to the incident window (inclusive)
df_window_trace = df_trace[(df_trace['ts_utc'] >= start_ts) & (df_trace['ts_utc'] <= end_ts)].copy()

# Services substrings to match
components = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
pattern = "(" + "|".join(re.escape(c) for c in components) + ")"

# Select only trace_name values that contain any of the substrings
mask_service = df_window_trace['trace_name'].str.contains(pattern, regex=True, na=False)
df_window_service = df_window_trace[mask_service].copy()

# Merge global p95 (computed from full file) into the windowed rows
df_window_service = df_window_service.merge(p95_trace, on='trace_name', how='left')

# Determine anomalies where value >= global p95_value (only when p95_value present)
mask_valid_p95 = df_window_service['p95_value'].notna()
df_window_service['is_anomaly'] = False
df_window_service.loc[mask_valid_p95, 'is_anomaly'] = df_window_service.loc[mask_valid_p95, 'value'] >= df_window_service.loc[mask_valid_p95, 'p95_value']

# Aggregate per trace_name within the window
grouped = df_window_service.groupby('trace_name', sort=False)
rows = []
for name, g in grouped:
    cnt = int(g.shape[0])
    an_cnt = int(g['is_anomaly'].sum())
    earliest_anom_ts = int(g.loc[g['is_anomaly'], 'timestamp'].min()) if an_cnt > 0 else pd.NA
    maxv = float(g['value'].max()) if cnt > 0 else np.nan
    p95v = float(g['p95_value'].iloc[0]) if pd.notna(g['p95_value'].iloc[0]) else np.nan
    ratio = (float(maxv) / float(p95v)) if (pd.notna(p95v) and p95v != 0) else pd.NA
    rows.append({
        'trace_name': name,
        'count_in_window': cnt,
        'anomaly_count': an_cnt,
        'earliest_anomaly_timestamp': earliest_anom_ts,
        'max_value_in_window': maxv,
        'p95_value': p95v,
        'ratio': ratio
    })

agg_trace_window = pd.DataFrame(rows)

# Normalize types and sort: ratio desc, then anomaly_count desc
if not agg_trace_window.empty:
    agg_trace_window['earliest_anomaly_timestamp'] = agg_trace_window['earliest_anomaly_timestamp'].astype('Int64')
    agg_trace_window['ratio_float'] = agg_trace_window['ratio'].apply(lambda x: float(x) if pd.notna(x) else np.nan)
    agg_trace_sorted = agg_trace_window.sort_values(by=['ratio_float','anomaly_count'], ascending=[False, False], na_position='last').drop(columns=['ratio_float']).reset_index(drop=True)
else:
    agg_trace_sorted = agg_trace_window.copy()

# Top 20 rows for compact output
top20_traces = agg_trace_sorted.head(20)[[
    'trace_name','count_in_window','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','p95_value','ratio'
]]

# Short list of trace_names with anomaly_count > 0 (limit 20)
anomalies_traces = agg_trace_sorted[agg_trace_sorted['anomaly_count']>0][
    ['trace_name','anomaly_count','earliest_anomaly_timestamp']
].head(20).reset_index(drop=True)

# Return compact results
top20_traces, anomalies_traces

```
Out[4]:
```
Summary (incident window 2024-01-22 09:53:25 — 2024-01-22 10:23:25 UTC)

High-level finding
- Traces showing the largest exceedances of their global P95s are primarily related to route-service, travel-service, order-service and auth-service. These exceedances are seen both in duration_p95/duration_mean metrics and in row_count (throughput) traces.

Top groups by ratio (max / global P95) — top 5
1. trace.to_ts-route-service.duration_p95 — ratio 1.595, anomalies=5, earliest_anomaly_timestamp=1705917540
2. trace.from_ts-travel-service.duration_p95 — ratio 1.442, anomalies=8, earliest_anomaly_timestamp=1705917420
3. trace.to_ts-travel-service.duration_p95 — ratio 1.341, anomalies=8, earliest_anomaly_timestamp=1705917360
4. trace.to_ts-auth-service.duration_p95 — ratio 1.218, anomalies=3, earliest_anomaly_timestamp=1705918260
5. trace.from_ts-auth-service.duration_p95 — ratio 1.144, anomalies=2, earliest_anomaly_timestamp=1705918500

Other notable items (selected)
- trace.to_ts-order-service.row_count — ratio 1.120, anomalies=9, earliest_anomaly_timestamp=1705917540 (max row_count = 549)
- Several travel-service traces (row_count, duration_mean, duration_p95) and order-service duration_p95/duration_mean also show repeated anomalies (anomaly_count 8–9).
- Many affected groups had anomaly_count between 2 and 9 (20 groups with anomaly_count>0 were identified in the result).

Interpretation / recommendation
- The most pronounced deviations are in duration_p95 metrics for route-service and travel-service and in row_count for order/travel — these indicate increased latency (higher tail durations) and elevated throughput spikes during the incident window.
- Prioritize investigating traces and services in this order: route-service (to_* duration_p95), travel-service (from/to duration_p95, row_count, duration_mean), order-service (row_count, duration_p95/duration_mean), and auth-service (duration_p95). Check related service logs, upstream/downstream latency and any capacity or routing changes around the earliest anomaly timestamps listed above.

If you want, I can convert the unix timestamps to human-readable UTC times or provide the full top-20 table and the full list of trace_names with anomaly details.

The original code execution output of IPython Kernel is also provided below for reference:

(                                    trace_name  count_in_window  anomaly_count  earliest_anomaly_timestamp  max_value_in_window    p95_value     ratio
0       trace.to_ts-route-service.duration_p95              100              5                  1705917540             0.006762     0.004239  1.595301
1    trace.from_ts-travel-service.duration_p95              150              8                  1705917420             0.092508     0.064171  1.441591
2      trace.to_ts-travel-service.duration_p95              149              8                  1705917360             1.012744     0.755399  1.340675
3        trace.to_ts-auth-service.duration_p95               50              3                  1705918260             0.808240     0.663520  1.218109
4      trace.from_ts-auth-service.duration_p95               25              2                  1705918500             0.019815     0.017315  1.144355
5          trace.to_ts-order-service.row_count              174              9                  1705917540           549.000000   490.350000  1.119608
6       trace.from_ts-travel-service.row_count              150              8                  1705917540          1888.000000  1690.650000   1.11673
7         trace.to_ts-travel-service.row_count              149              8                  1705917540          1888.000000  1690.800000  1.116631
8     trace.to_ts-travel-service.duration_mean              149              8                  1705917540             0.622816     0.563310  1.105636
9       trace.to_ts-order-service.duration_p95              174              9                  1705917660             1.099157     1.025555  1.071768
10  trace.from_ts-travel-service.duration_mean              150              8                  1705917360             0.051773     0.048398  1.069736
11     trace.to_ts-order-service.duration_mean              174              9                  1705917600             0.996727     0.936283  1.064557
12      trace.to_ts-train-service.duration_p95              100              5                  1705917600             0.006396     0.006069   1.05379
13    trace.from_ts-order-service.duration_p95               50              3                  1705917660             1.076594     1.021840  1.053584
14         trace.to_ts-train-service.row_count              100              5                  1705917540          2860.000000  2740.600000  1.043567
15         trace.to_ts-route-service.row_count              100              5                  1705917540          2264.000000  2186.500000  1.035445
16     trace.to_ts-route-service.duration_mean              100              5                  1705917600             0.003622     0.003500  1.034907
17       trace.from_ts-order-service.row_count               50              3                  1705917540           549.000000   531.050000  1.033801
18   trace.from_ts-order-service.duration_mean               50              3                  1705918080             0.970390     0.943239  1.028785
19       trace.from_ts-train-service.row_count               25              2                  1705917540          2860.000000  2783.200000  1.027594,                                     trace_name  anomaly_count  earliest_anomaly_timestamp
0       trace.to_ts-route-service.duration_p95              5                  1705917540
1    trace.from_ts-travel-service.duration_p95              8                  1705917420
2      trace.to_ts-travel-service.duration_p95              8                  1705917360
3        trace.to_ts-auth-service.duration_p95              3                  1705918260
4      trace.from_ts-auth-service.duration_p95              2                  1705918500
5          trace.to_ts-order-service.row_count              9                  1705917540
6       trace.from_ts-travel-service.row_count              8                  1705917540
7         trace.to_ts-travel-service.row_count              8                  1705917540
8     trace.to_ts-travel-service.duration_mean              8                  1705917540
9       trace.to_ts-order-service.duration_p95              9                  1705917660
10  trace.from_ts-travel-service.duration_mean              8                  1705917360
11     trace.to_ts-order-service.duration_mean              9                  1705917600
12      trace.to_ts-train-service.duration_p95              5                  1705917600
13    trace.from_ts-order-service.duration_p95              3                  1705917660
14         trace.to_ts-train-service.row_count              5                  1705917540
15         trace.to_ts-route-service.row_count              5                  1705917540
16     trace.to_ts-route-service.duration_mean              5                  1705917600
17       trace.from_ts-order-service.row_count              3                  1705917540
18   trace.from_ts-order-service.duration_mean              3                  1705918080
19       trace.from_ts-train-service.row_count              2                  1705917540)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse previously computed agg_window_sorted (per-group windowed aggregates) which should exist in the kernel.
# It contains columns including: cmdb_id, kpi_name, anomaly_count, earliest_anomaly_timestamp, ratio
try:
    agg_window_sorted
except NameError:
    raise RuntimeError("Required variable 'agg_window_sorted' not found in the kernel. Please run the prior anomaly aggregation step first.")

# Select only groups that had anomalies (anomaly_count > 0)
faulty_groups = agg_window_sorted[agg_window_sorted['anomaly_count'] > 0].copy()

# Ensure earliest_anomaly_timestamp is Int64 NA-able, ratio is numeric
faulty_groups['earliest_anomaly_timestamp'] = faulty_groups['earliest_anomaly_timestamp'].astype('Int64')
faulty_groups['ratio'] = pd.to_numeric(faulty_groups['ratio'], errors='coerce')

# Aggregate by cmdb_id
def build_top_kpis(subdf, top_n=3):
    sub = subdf.sort_values('ratio', ascending=False).head(top_n)
    items = []
    for _, r in sub.iterrows():
        if pd.notna(r['ratio']):
            items.append(f"{r['kpi_name']} ({r['ratio']:.3f})")
        else:
            items.append(f"{r['kpi_name']} (NA)")
    return ", ".join(items)

rows = []
grouped = faulty_groups.groupby('cmdb_id', sort=True)
for cmdb, g in grouped:
    num_faulty = int(g.shape[0])
    # earliest_fault_ts: minimum earliest_anomaly_timestamp among that service's groups (NA-safe)
    if g['earliest_anomaly_timestamp'].notna().any():
        earliest_ts = int(g['earliest_anomaly_timestamp'].dropna().min())
    else:
        earliest_ts = pd.NA
    max_ratio = float(g['ratio'].dropna().max()) if g['ratio'].dropna().size > 0 else pd.NA
    top_kpis = build_top_kpis(g, top_n=3)
    rows.append({
        'cmdb_id': cmdb,
        'num_faulty_groups': num_faulty,
        'earliest_fault_ts': earliest_ts,
        'max_ratio': max_ratio,
        'top_kpis': top_kpis
    })

service_summary = pd.DataFrame(rows)

# Normalize types
service_summary['earliest_fault_ts'] = service_summary['earliest_fault_ts'].astype('Int64')

# Limit to up to 10 rows (we expect 5 services)
service_summary_limited = service_summary.sort_values(['cmdb_id']).head(10).reset_index(drop=True)

# Prepare separate short sorted lists:
# 1) by earliest_fault_ts ascending (earliest first)
earliest_sorted = service_summary.sort_values(by=['earliest_fault_ts', 'cmdb_id'], ascending=[True, True])
earliest_sorted = earliest_sorted[['cmdb_id','num_faulty_groups','earliest_fault_ts','max_ratio']].reset_index(drop=True).head(5)

# 2) by max_ratio descending (severity)
severity_sorted = service_summary.sort_values(by=['max_ratio','cmdb_id'], ascending=[False, True])
severity_sorted = severity_sorted[['cmdb_id','num_faulty_groups','earliest_fault_ts','max_ratio']].reset_index(drop=True).head(5)

# Return compact outputs (display variables)
service_summary_limited, earliest_sorted, severity_sorted

```
Out[5]:
```
```python
# Build a plain-English summary string from the aggregated service results
import pandas as pd

# Reuse existing variables in the kernel
svc_df = service_summary_limited
earliest_df = earliest_sorted
severity_df = severity_sorted

def ts_iso(ts):
    if pd.isna(ts):
        return "None"
    return pd.to_datetime(int(ts), unit='s', utc=True).strftime("%Y-%m-%d %H:%M:%S UTC")

lines = []
lines.append("Summary of faulty services from metric-group anomalies (window 2024-01-22 09:53:25 — 2024-01-22 10:23:25 UTC):")
lines.append("")
# Per-service summary (compact)
for _, r in svc_df.iterrows():
    lines.append(
        f"- {r['cmdb_id']}: num_faulty_groups={int(r['num_faulty_groups'])}, "
        f"earliest_fault_ts={ts_iso(r['earliest_fault_ts'])}, "
        f"max_ratio={float(r['max_ratio']):.3f}, top_kpis={r['top_kpis']}"
    )

lines.append("")
# Earliest onset list (earliest first)
lines.append("Services by earliest fault onset (earliest first):")
for _, r in earliest_df.iterrows():
    lines.append(
        f"- {r['cmdb_id']}: earliest_fault_ts={ts_iso(r['earliest_fault_ts'])}, "
        f"num_faulty_groups={int(r['num_faulty_groups'])}, max_ratio={float(r['max_ratio']):.3f}"
    )

lines.append("")
# Severity list (by max_ratio desc)
lines.append("Services by severity (max_ratio, highest first):")
for _, r in severity_df.iterrows():
    lines.append(
        f"- {r['cmdb_id']}: max_ratio={float(r['max_ratio']):.3f}, "
        f"num_faulty_groups={int(r['num_faulty_groups'])}, earliest_fault_ts={ts_iso(r['earliest_fault_ts'])}"
    )

lines.append("")
lines.append("Key takeaways:")
lines.append("- ts-order-service shows the largest deviation (max_ratio ~8.65) and multiple faulty KPI groups (8 groups), with earliest anomalies at 2024-01-22 09:56:00 UTC.")
lines.append("- ts-auth-service also shows notable deviation (max_ratio ~2.16), but its earliest anomaly happened later (~2024-01-22 10:08:00 UTC).")
lines.append("- ts-travel-service, ts-route-service and ts-train-service have smaller max_ratios (~1.05–1.24) but multiple faulty groups; travel and order show the earliest onsets.")
lines.append("- Recommendation: prioritize investigation on ts-order-service (highest severity) and ts-travel-service/ts-route-service for early onset patterns, then ts-auth-service for its high ratio.")

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id  num_faulty_groups  earliest_fault_ts  max_ratio                                           top_kpis
0    ts-auth-service                  7         1705918080   2.158322  diskio (2.158), latency-90 (1.054), latency-50...
1   ts-order-service                  8         1705917360   8.653825  latency-90 (8.654), workload (1.080), diskio (...
2   ts-route-service                  7         1705917540   1.052816    diskio (1.053), cpu (1.042), latency-50 (1.015)
3   ts-train-service                  7         1705917600   1.050948      cpu (1.051), diskio (1.025), workload (1.018)
4  ts-travel-service                  8         1705917360   1.244441    cpu (1.244), latency-90 (1.115), socket (1.032),              cmdb_id  num_faulty_groups  earliest_fault_ts  max_ratio
0   ts-order-service                  8         1705917360   8.653825
1  ts-travel-service                  8         1705917360   1.244441
2   ts-route-service                  7         1705917540   1.052816
3   ts-train-service                  7         1705917600   1.050948
4    ts-auth-service                  7         1705918080   2.158322,              cmdb_id  num_faulty_groups  earliest_fault_ts  max_ratio
0   ts-order-service                  8         1705917360   8.653825
1    ts-auth-service                  7         1705918080   2.158322
2  ts-travel-service                  8         1705917360   1.244441
3   ts-route-service                  7         1705917540   1.052816
4   ts-train-service                  7         1705917600   1.050948)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse existing variables if present, otherwise load CSVs
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
try:
    df_error
except NameError:
    df_error = pd.read_csv("error_logs.csv")

# Ensure timestamp -> UTC datetimes (rule 10)
df_log['ts_utc'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_error['ts_utc'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# Components and incident window (UTC)
components = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
start_ts = pd.Timestamp("2024-01-22 09:53:25", tz="UTC")
end_ts   = pd.Timestamp("2024-01-22 10:23:25", tz="UTC")

# 1) Log.csv: compute global P95 per (cmdb_id, log_name) across full file (rule 9)
p95_log = (
    df_log
    .groupby(['cmdb_id','log_name'], sort=False, as_index=False)
    .agg(p95_value=('value', lambda s: float(s.quantile(0.95))))
)

# Filter log rows to the five components and the incident window (inclusive)
df_log_window = df_log[
    df_log['cmdb_id'].isin(components) &
    (df_log['ts_utc'] >= start_ts) &
    (df_log['ts_utc'] <= end_ts)
].copy()

# Merge global p95 into windowed logs
df_log_window = df_log_window.merge(p95_log, on=['cmdb_id','log_name'], how='left')

# Mark anomalies where value >= global p95 (only when p95 present)
df_log_window['is_anomaly'] = False
mask_valid = df_log_window['p95_value'].notna()
df_log_window.loc[mask_valid, 'is_anomaly'] = df_log_window.loc[mask_valid, 'value'] >= df_log_window.loc[mask_valid, 'p95_value']

# Aggregate per (cmdb_id, log_name) within the window
rows = []
grouped = df_log_window.groupby(['cmdb_id','log_name'], sort=False)
for (cmdb, logname), g in grouped:
    cnt = int(g.shape[0])
    an_cnt = int(g['is_anomaly'].sum())
    earliest_anom_ts = int(g.loc[g['is_anomaly'], 'timestamp'].min()) if an_cnt > 0 else pd.NA
    maxv = float(g['value'].max()) if cnt > 0 else np.nan
    p95v = float(g['p95_value'].iloc[0]) if pd.notna(g['p95_value'].iloc[0]) else np.nan
    ratio = (float(maxv) / float(p95v)) if (pd.notna(p95v) and p95v != 0) else pd.NA
    rows.append({
        'cmdb_id': cmdb,
        'log_name': logname,
        'count_in_window': cnt,
        'anomaly_count': an_cnt,
        'earliest_anomaly_timestamp': earliest_anom_ts,
        'max_value_in_window': maxv,
        'p95_value': p95v,
        'ratio': ratio
    })

logs_agg = pd.DataFrame(rows)

# Normalize types and sort by ratio desc then anomaly_count desc
if not logs_agg.empty:
    logs_agg['earliest_anomaly_timestamp'] = logs_agg['earliest_anomaly_timestamp'].astype('Int64')
    logs_agg['ratio_float'] = logs_agg['ratio'].apply(lambda x: float(x) if pd.notna(x) else np.nan)
    logs_sorted = logs_agg.sort_values(by=['ratio_float','anomaly_count'], ascending=[False, False], na_position='last').drop(columns=['ratio_float']).reset_index(drop=True)
else:
    logs_sorted = logs_agg.copy()

# Top 20 rows (compact)
logs_top20 = logs_sorted.head(20)[[
    'cmdb_id','log_name','count_in_window','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','p95_value','ratio'
]]

# 2) error_logs.csv: filter to components and incident window
df_error_window = df_error[
    df_error['cmdb_id'].isin(components) &
    (df_error['ts_utc'] >= start_ts) &
    (df_error['ts_utc'] <= end_ts)
].copy()

# Select up to 20 rows showing timestamp, cmdb_id, message
if not df_error_window.empty:
    error_head = df_error_window[['timestamp','cmdb_id','message']].head(20).reset_index(drop=True)
else:
    # Empty DataFrame with the requested columns
    error_head = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Return compact outputs (two DataFrames)
logs_top20, error_head

```
Out[6]:
```
- Summary of log metrics (incident window 2024-01-22 09:53:25 — 2024-01-22 10:23:25 UTC):

  - log.row_count had small but measurable spikes above its global P95:
    - ts-order-service: max 482 vs P95 459 → ratio ≈ 1.05, anomaly_count = 2, earliest anomaly ≈ 2024-01-22 09:59:00 UTC.
    - ts-route-service: max 1132 vs P95 1102.8 → ratio ≈ 1.03, anomaly_count = 2, earliest ≈ 2024-01-22 09:59:00 UTC.
    - ts-travel-service: max 2104 vs P95 2058.2 → ratio ≈ 1.02, anomaly_count = 2, earliest ≈ 2024-01-22 09:59:00 UTC.
    - ts-auth-service: max 354 vs P95 352 → ratio ≈ 1.01, anomaly_count = 3, earliest ≈ 2024-01-22 09:59:00 UTC.
    - These represent modest throughput spikes (order-service is largest).

  - log.error_count shows anomaly_count = 25 for multiple services but both max_value_in_window and global P95 are 0, so ratio is undefined. This is not a meaningful non-zero error spike—it reflects the metric being zero and the ≥ P95 logic flagging equality. Treat these with caution.

- error_logs.csv:
  - No error log rows were found for the five services in the incident window (empty result).

- Takeaway / recommended next steps:
  - Prioritize investigating ts-order-service (largest relative increase in log.row_count) around ~09:59 UTC.
  - Verify whether the small row_count spikes correspond to real traffic bursts or instrumentation/aggregation artifacts.
  - Ignore log.error_count anomalies as actionable alerts unless you find non-zero error messages in raw logs—error_logs had no entries for these services in the window.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id         log_name  count_in_window  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  p95_value     ratio
0   ts-order-service    log.row_count               25              2                  1705917540                482.0      459.0  1.050109
1   ts-route-service    log.row_count               25              2                  1705917540               1132.0     1102.8  1.026478
2  ts-travel-service    log.row_count               25              2                  1705917540               2104.0     2058.2  1.022252
3    ts-auth-service    log.row_count               25              3                  1705917540                354.0      352.0  1.005682
4    ts-auth-service  log.error_count               25             25                  1705917360                  0.0        0.0      <NA>
5   ts-order-service  log.error_count               25             25                  1705917360                  0.0        0.0      <NA>
6   ts-route-service  log.error_count               25             25                  1705917360                  0.0        0.0      <NA>
7  ts-travel-service  log.error_count               25             25                  1705917360                  0.0        0.0      <NA>, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[7]:

import pandas as pd
import re

# Reuse df_trace if available, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Ensure timestamp UTC datetime column exists (rule 10)
if 'ts_utc' not in df_trace.columns:
    df_trace['ts_utc'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Candidate services and regex for substring match
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
cand_pattern = "(" + "|".join(re.escape(c) for c in candidates) + ")"

# Primary filter: trace_name containing BOTH 'from_ts-' and 'to_ts-' and any candidate service substring
mask_both = (
    df_trace['trace_name'].str.contains('from_ts-', na=False) &
    df_trace['trace_name'].str.contains('to_ts-', na=False) &
    df_trace['trace_name'].str.contains(cand_pattern, regex=True, na=False)
)

primary_names = df_trace.loc[mask_both, 'trace_name'].unique()

# Aggregate function
def aggregate_trace_names(names):
    if len(names) == 0:
        return pd.DataFrame(columns=['trace_name','total_row_count','min_timestamp','max_timestamp'])
    sub = df_trace[df_trace['trace_name'].isin(names)].copy()
    agg = (
        sub
        .groupby('trace_name', sort=False, as_index=False)
        .agg(
            total_row_count=('trace_name', 'size'),
            min_timestamp=('timestamp', 'min'),
            max_timestamp=('timestamp', 'max')
        )
        .sort_values('total_row_count', ascending=False)
        .head(50)
        .reset_index(drop=True)
    )
    # Ensure integer timestamps
    agg['min_timestamp'] = agg['min_timestamp'].astype('Int64')
    agg['max_timestamp'] = agg['max_timestamp'].astype('Int64')
    return agg

primary_matches = aggregate_trace_names(primary_names)

# If primary is empty, also compute fallback: trace_name containing EITHER 'from_ts-' OR 'to_ts-' and any candidate service
if primary_matches.shape[0] == 0:
    mask_either = (
        (df_trace['trace_name'].str.contains('from_ts-', na=False) |
         df_trace['trace_name'].str.contains('to_ts-', na=False)) &
        df_trace['trace_name'].str.contains(cand_pattern, regex=True, na=False)
    )
    fallback_names = df_trace.loc[mask_either, 'trace_name'].unique()
    fallback_matches = aggregate_trace_names(fallback_names)
else:
    # empty fallback DF for clarity
    fallback_matches = pd.DataFrame(columns=['trace_name','total_row_count','min_timestamp','max_timestamp'])

# Return results: primary_matches (preferred), fallback_matches (if primary empty this will contain results)
primary_matches, fallback_matches

```
Out[7]:
```
- No trace names contained both substrings "from_ts-" AND "to_ts-" together for the candidate services — the strict "both" match returned zero results.

- However, a fallback scan for trace names containing either "from_ts-" OR "to_ts-" plus any of the five candidate services returned 40 matching trace_name values (examples below). These represent direct incoming ("from_ts-...") or outgoing ("to_ts-...") call-chain traces involving the candidate services.

- Overall coverage for those matches:
  - Number of distinct matching trace_name values: 40
  - Timestamp range across these trace names (full trace.csv): min = 1705917360, max = 1705918800 (Unix seconds, UTC)

- Top examples by total row count (most frequent trace_name entries in the full trace.csv):
  1. trace.to_ts-order-service.row_count — 174 rows
  2. trace.to_ts-order-service.error_rate — 174 rows
  3. trace.to_ts-order-service.duration_p95 — 174 rows
  4. trace.to_ts-order-service.duration_mean — 174 rows
  5. trace.from_ts-travel-service.row_count — 150 rows
  (There are many other matches for to_/from_ traces involving travel, train, route, auth services — total 40 distinct trace_name values; examples include to_ts-travel-service.*, from_ts-travel-service.*, to_ts-route-service.*, from_ts-route-service.*, from_ts-train-service.*.)

- Interpretation / next step:
  - There are many per-service directional trace metrics (to_*/from_*) involving the candidate services, but no single trace_name string combined both "from_ts-" and "to_ts-". To analyze call chains between specific pairs of services, inspect the listed to_ts-* and from_ts-* trace_name entries (especially the top ones by row count) and correlate timestamps/trace IDs across the traces to reconstruct direct call relationships.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [trace_name, total_row_count, min_timestamp, max_timestamp]
Index: [],                                    trace_name  total_row_count  min_timestamp  max_timestamp
0         trace.to_ts-order-service.row_count              174     1705917360     1705918800
1        trace.to_ts-order-service.error_rate              174     1705917360     1705918800
2      trace.to_ts-order-service.duration_p95              174     1705917360     1705918800
3     trace.to_ts-order-service.duration_mean              174     1705917360     1705918800
4      trace.from_ts-travel-service.row_count              150     1705917360     1705918800
..                                        ...              ...            ...            ...
35      trace.from_ts-route-service.row_count               25     1705917360     1705918800
36  trace.from_ts-train-service.duration_mean               25     1705917360     1705918800
37   trace.from_ts-train-service.duration_p95               25     1705917360     1705918800
38     trace.from_ts-train-service.error_rate               25     1705917360     1705918800
39      trace.from_ts-train-service.row_count               25     1705917360     1705918800

[40 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np

# Reuse df_trace if present, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Compute global P95 per trace_name across full file (rule 9)
p95_trace = (
    df_trace
    .groupby('trace_name', sort=False, as_index=False)
    .agg(p95_value=('value', lambda s: float(s.quantile(0.95))))
)

# Map of p95 by trace_name
p95_map = p95_trace.set_index('trace_name')['p95_value'].to_dict()

# Services and targeted directional row_count trace names
services = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
from_names = {s: f"trace.from_ts-{s}.row_count" for s in services}
to_names = {s: f"trace.to_ts-{s}.row_count" for s in services}

target_names = list(from_names.values()) + list(to_names.values())

# Ensure timestamp UTC datetime column exists and filter window using UTC (rule 10)
if 'ts_utc' not in df_trace.columns:
    df_trace['ts_utc'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

start_ts = pd.Timestamp("2024-01-22 09:53:25", tz="UTC")
end_ts   = pd.Timestamp("2024-01-22 10:23:25", tz="UTC")

df_window = df_trace[
    (df_trace['trace_name'].isin(target_names)) &
    (df_trace['ts_utc'] >= start_ts) &
    (df_trace['ts_utc'] <= end_ts)
].copy()

# Pivot so rows are timestamps and columns are the directed row_count trace values
if df_window.empty:
    overlaps_df = pd.DataFrame(columns=['from_service','to_service','overlap_count','example_overlap_timestamps'])
else:
    pivot = df_window.pivot_table(index='timestamp', columns='trace_name', values='value', aggfunc='first')

    rows = []
    for from_s, from_name in from_names.items():
        for to_s, to_name in to_names.items():
            if from_s == to_s:
                continue
            # Skip if either p95 is missing
            p95_from = p95_map.get(from_name, np.nan)
            p95_to = p95_map.get(to_name, np.nan)
            if pd.isna(p95_from) or pd.isna(p95_to):
                overlap_count = 0
                example_ts = []
            else:
                # Both columns must exist in pivot
                if from_name not in pivot.columns or to_name not in pivot.columns:
                    overlap_count = 0
                    example_ts = []
                else:
                    mask = (pivot[from_name] >= p95_from) & (pivot[to_name] >= p95_to)
                    # drop NA rows where one side is missing
                    mask = mask & pivot[from_name].notna() & pivot[to_name].notna()
                    overlap_timestamps = pivot.index[mask].tolist()
                    overlap_count = len(overlap_timestamps)
                    example_ts = overlap_timestamps[:10]
            rows.append({
                'from_service': from_s,
                'to_service': to_s,
                'overlap_count': int(overlap_count),
                'example_overlap_timestamps': example_ts
            })

    overlaps_df = pd.DataFrame(rows).sort_values(by='overlap_count', ascending=False).reset_index(drop=True).head(10)

# Display compact result (top 10 pairs)
overlaps_df

```
Out[8]:
```
Result: No overlapping high row_count events found.

- For the incident window 2024-01-22 09:53:25 — 2024-01-22 10:23:25 UTC and for the five services (ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service), there are zero ordered pairs (from A → to B, with A != B) where, at the same timestamp, both:
  - trace.from_ts-A.row_count ≥ global P95(for that from-trace) and
  - trace.to_ts-B.row_count ≥ global P95(for that to-trace).

- In other words, no service pair showed simultaneous row_count spikes above their per-trace global 95th percentiles during the incident window.

If you want next steps, I can: relax the threshold (e.g., >= 90th percentile), inspect individual pairs or raw timestamps for near-misses, or show the per-trace pivot to verify alignment/missing data.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [from_service, to_service, overlap_count, example_overlap_timestamps]
Index: []```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np

# Reuse df_trace if already loaded, otherwise load it
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Ensure timestamp UTC datetime column exists (rule 10)
if 'ts_utc' not in df_trace.columns:
    df_trace['ts_utc'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Services and target trace name patterns
services = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
from_names = {s: f"trace.from_ts-{s}.duration_p95" for s in services}
to_names = {s: f"trace.to_ts-{s}.duration_p95" for s in services}
target_names = list(from_names.values()) + list(to_names.values())

# 1) Compute global P95 per trace_name across the full file (must be done before filtering) (rule 9)
p95_trace = (
    df_trace
    .groupby('trace_name', sort=False, as_index=False)
    .agg(p95_value=('value', lambda s: float(s.quantile(0.95))))
)
p95_map = p95_trace.set_index('trace_name')['p95_value'].to_dict()

# 2) Filter trace rows to the incident window (inclusive) and to our target trace names
start_ts = pd.Timestamp("2024-01-22 09:53:25", tz="UTC")
end_ts   = pd.Timestamp("2024-01-22 10:23:25", tz="UTC")

df_window = df_trace[
    (df_trace['ts_utc'] >= start_ts) &
    (df_trace['ts_utc'] <= end_ts) &
    (df_trace['trace_name'].isin(target_names))
].copy()

# 3) Pivot window rows so index = timestamp (unix seconds), columns = trace_name, values = value
if df_window.empty:
    overlaps_df = pd.DataFrame(columns=['from_service','to_service','overlap_count','example_overlap_timestamps'])
else:
    pivot = df_window.pivot_table(index='timestamp', columns='trace_name', values='value', aggfunc='first')

    rows = []
    for from_s, from_name in from_names.items():
        for to_s, to_name in to_names.items():
            if from_s == to_s:
                continue
            p95_from = p95_map.get(from_name, np.nan)
            p95_to = p95_map.get(to_name, np.nan)
            # If either p95 missing, overlap is zero
            if pd.isna(p95_from) or pd.isna(p95_to):
                overlap_count = 0
                example_ts = []
            else:
                if from_name not in pivot.columns or to_name not in pivot.columns:
                    overlap_count = 0
                    example_ts = []
                else:
                    mask = (pivot[from_name] >= p95_from) & (pivot[to_name] >= p95_to)
                    mask = mask & pivot[from_name].notna() & pivot[to_name].notna()
                    overlap_timestamps = pivot.index[mask].tolist()
                    overlap_count = len(overlap_timestamps)
                    example_ts = overlap_timestamps[:10]
            rows.append({
                'from_service': from_s,
                'to_service': to_s,
                'overlap_count': int(overlap_count),
                'example_overlap_timestamps': example_ts
            })

    overlaps_df = pd.DataFrame(rows).sort_values(by='overlap_count', ascending=False).reset_index(drop=True).head(10)

# Final compact output
overlaps_df

```
Out[9]:
```
```python
# Plain-English summary of the overlap check results
summary_text = (
    "Result summary:\n"
    "- Window: 2024-01-22 09:53:25 UTC through 2024-01-22 10:23:25 UTC (inclusive).\n"
    "- Focus: trace.from_ts-<service>.duration_p95 and trace.to_ts-<service>.duration_p95 for services "
    "ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service.\n\n"
    "Finding:\n"
    "- There are ZERO ordered service pairs (from A → to B, A != B) with any timestamps in the incident window "
    "where BOTH the from-A.duration_p95 AND the to-B.duration_p95 values simultaneously met or exceeded their "
    "respective global P95 thresholds.\n\n"
    "Interpretation and short next steps:\n"
    "- No simultaneous tail-latency spikes (by the >= global P95 criterion) were observed across different services "
    "during the incident window. This suggests that cross-service paired duration_p95 spikes did not coincide at the "
    "same minute-level timestamps.\n"
    "- If you want to catch near-misses or weaker correlations, consider relaxing the threshold (e.g., >= 90th percentile), "
    "looking for one-side leads/lags, or inspecting per-service duration_p95 anomalies and their timestamps for causal ordering.\n"
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [from_service, to_service, overlap_count, example_overlap_timestamps]
Index: []```
```